# Original coursework database workflow
Archive of PostgreSQL + MongoDB + Flask workflow. Database credentials are environment variables. Some cells rebuild/drop project tables: use an isolated development database only. This archive is not executed by the standalone explorer. Use src/build_demo.py for a database-free portfolio export.

# Connect Database

In [ ]:
import os
import psycopg
conn = psycopg.connect(
    host="localhost",
    port="5432",
    dbname="final", 
    user="postgres",
    password=os.environ["POSTGRES_PASSWORD"] 
)

In [ ]:
cur = conn.cursor()
print("Connected to PostgreSQL")

cur.execute("""
    DROP TABLE IF EXISTS daily_price;
    DROP TABLE IF EXISTS company;
""")

# Create Tech Company Table

In [ ]:
cur.execute("""
    CREATE TABLE company (
        company_id      SERIAL PRIMARY KEY, 
        cik             VARCHAR(10) NOT NULL UNIQUE,
        ticker          VARCHAR(10) NOT NULL UNIQUE,
        company_name    TEXT NOT NULL,
        sic             INTEGER,
        sic_description TEXT
    );""")
conn.commit()
print("Table 'company' created.")

In [ ]:
import pandas as pd
df = pd.read_csv("primary_ticker_only.csv")
df.head()

df_clean = df.rename(columns={
    "ticker": "ticker",
    "title": "company_name",
    "CIK": "cik",
    "SIC": "sic",
    "SIC_description": "sic_description"
})

# Convert CIK to 10 digits
df_clean["cik"] = df_clean["cik"].astype(str).str.zfill(10)
df_clean.head()

In [ ]:
for _, row in df_clean.iterrows():
    cur.execute("""
        INSERT INTO company (cik, ticker, company_name, sic, sic_description)
        VALUES (%s, %s, %s, %s, %s);
    """, (
        row["cik"],
        row["ticker"],
        row["company_name"],
        int(row["sic"]) if not pd.isna(row["sic"]) else None,
        row["sic_description"]
    ))

conn.commit()
print("All companies inserted!")

In [ ]:
cur.execute("SELECT company_id, cik, ticker, company_name FROM company LIMIT 10;")
for row in cur.fetchall():
    print(row)
cur.execute("SELECT COUNT(*) FROM company;")
print("Total companies:", cur.fetchone()[0])

cur.execute("""
    SELECT
        COUNT(*) FILTER (WHERE cik IS NULL)              AS cik_nulls,
        COUNT(*) FILTER (WHERE ticker IS NULL)           AS ticker_nulls,
        COUNT(*) FILTER (WHERE company_name IS NULL)     AS company_name_nulls,
        COUNT(*) FILTER (WHERE sic IS NULL)              AS sic_nulls,
        COUNT(*) FILTER (WHERE sic_description IS NULL)  AS sic_description_nulls
    FROM company;
""")

print("NULL count per column:", cur.fetchone())

# Create daily_price table & insert data from yahoo finance

In [ ]:
!pip install -U "psycopg[binary]" yfinance pandas

In [ ]:
import os
import psycopg

conn = psycopg.connect(
    host="localhost",
    port="5432",
    dbname="final",
    user="postgres",
    password=os.environ["POSTGRES_PASSWORD"]
)
cur = conn.cursor()
print("Connected to Final database.")

In [ ]:
cur.execute("""
    DROP TABLE IF EXISTS daily_price;

    CREATE TABLE daily_price (
        company_id  INTEGER NOT NULL REFERENCES company(company_id),
        trade_date  DATE    NOT NULL,
        open        NUMERIC(18,4),
        high        NUMERIC(18,4),
        low         NUMERIC(18,4),
        close       NUMERIC(18,4),
        adj_close   NUMERIC(18,4),
        volume      BIGINT,
        PRIMARY KEY (company_id, trade_date)
    );
""")
conn.commit()

print("Table 'daily_price' created.")

In [ ]:
import pandas as pd

df_comp = pd.read_sql("SELECT company_id, ticker FROM company ORDER BY company_id;", conn)

print("Total companies:", len(df_comp))
df_comp.head()

Set the starting point as November 28, 2015 and the ending point as today. Daily OHLCV of grabbing a certain ticker from FIXED_START to TODAY. Take as much as you have. If the company has been listed for less than ten years, the returned data will automatically decrease.

In [ ]:
import yfinance as yf
from datetime import datetime

FIXED_START = "2015-11-28"                       
# fixed beginning date
TODAY = datetime.today().strftime("%Y-%m-%d")    
# end date = today

print("Fetching window:", FIXED_START, "→", TODAY)

def fetch_price_fixed_window(ticker: str):

    try:
        df = yf.download(
            ticker,
            start=FIXED_START,
            end=TODAY,
            auto_adjust=False,
            progress=False
        )

        if df.empty:
            print(f"⚠️ {ticker}: no price data found.")
            return None

        df.reset_index(inplace=True)

        df.rename(columns={
            "Date": "trade_date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adj_close",
            "Volume": "volume"
        }, inplace=True)

        df = df[["trade_date", "open", "high", "low", "close", "adj_close", "volume"]]

        return df

    except Exception as e:
        print(f"❌ Error downloading {ticker}: {e}")
        return None

rec is a tuple, corresponding to the column order:  
(trade_date, open, high, low, close, adj_close, volume)

In [ ]:
import math

for _, row in df_comp.iterrows():
    cid = int(row["company_id"])
    ticker = row["ticker"]

    print(f"\nFetching from {FIXED_START} to {TODAY} for {ticker} (company_id={cid}) ...")

    df_price = fetch_price_fixed_window(ticker)
    if df_price is None:
        continue

    print("  Columns from yfinance:", list(df_price.columns))

    for rec in df_price.itertuples(index=False, name=None):
        
        trade_date_raw, open_raw, high_raw, low_raw, close_raw, adj_close_raw, vol_raw = rec

        if isinstance(trade_date_raw, pd.Timestamp):
            trade_date = trade_date_raw.date()
        else:
            trade_date = trade_date_raw

        def to_num(x):
            if x is None:
                return None
            if isinstance(x, float) and math.isnan(x):
                return None
            return float(x)

        open_      = to_num(open_raw)
        high_      = to_num(high_raw)
        low_       = to_num(low_raw)
        close_     = to_num(close_raw)
        adj_close_ = to_num(adj_close_raw)

        vol = vol_raw
        if vol is None or (isinstance(vol, float) and math.isnan(vol)):
            vol = None
        else:
            vol = int(vol)

        cur.execute("""
            INSERT INTO daily_price
                (company_id, trade_date, open, high, low, close, adj_close, volume)
            VALUES
                (%s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (company_id, trade_date) DO NOTHING;
        """, (
            cid,
            trade_date,
            open_,
            high_,
            low_,
            close_,
            adj_close_,
            vol
        ))

    conn.commit()
    print(f"✔ Inserted {len(df_price)} rows for {ticker}.")

In [ ]:
cur.execute("""
    SELECT company_id, trade_date, open, close
    FROM daily_price
    ORDER BY trade_date DESC
    LIMIT 10;
""")
rows = cur.fetchall()

print("\nSample rows from daily_price:")
for r in rows:
    print(r)

Check the data download status

In [ ]:
print("\n" + "="*50)
print("Data download status check:")
print("="*50)

cur.execute("""
    SELECT COUNT(DISTINCT company_id) 
    FROM daily_price;
""")
companies_with_data = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM company;")
total_companies = cur.fetchone()[0]

companies_without_data = total_companies - companies_with_data

print(f"Number of total comapnies: {total_companies}")
print(f"Number of companies with data: {companies_with_data}")
print(f"Number of companies without data: {companies_without_data}")
print(f"Data coverage rate: {companies_with_data/total_companies*100:.1f}%")

if companies_without_data > 0:
    print(f"\nThe list of companies without data:")
    cur.execute("""
        SELECT c.company_id, c.ticker, c.company_name
        FROM company c
        LEFT JOIN daily_price dp ON c.company_id = dp.company_id
        WHERE dp.company_id IS NULL
        ORDER BY c.ticker;
    """)
    
    no_data_companies = cur.fetchall()
    for company_id, ticker, name in no_data_companies[:10]:
        print(f"  {ticker}: {name}")
    
    if len(no_data_companies) > 10:
        print(f"  ... There are {len(no_data_companies)-10}  more companies")

print(f"\nData volume statistics:")
cur.execute("""
    SELECT 
        COUNT(DISTINCT company_id) as companies,
        MIN(cnt) as min_records,
        MAX(cnt) as max_records,
        AVG(cnt)::INT as avg_records
    FROM (
        SELECT company_id, COUNT(*) as cnt
        FROM daily_price 
        GROUP BY company_id
    ) t;
""")

stats = cur.fetchone()
print(f"The number of companies with data: {stats[0]}")
print(f"Minimum number of records: {stats[1]}")
print(f"Maximum number of records: {stats[2]}")  
print(f"Average number of records: {stats[3]}")

# Create sec_fillings table & insert data from SEC EDGAR

In [ ]:
!pip install -U "psycopg[binary]" pymongo requests beautifulsoup4 pandas

In [ ]:
import os
import psycopg
from pymongo import MongoClient
import requests
from bs4 import BeautifulSoup
from datetime import date
import time

pg_conn = psycopg.connect(
    host="localhost",
    port="5432",
    dbname="final", 
    user="postgres",
    password=os.environ["POSTGRES_PASSWORD"]
)
pg_cur = pg_conn.cursor()
print("Connected to PostgreSQL 'final'")

In [ ]:
pg_cur.execute("""
    SELECT company_id, cik, ticker, company_name
    FROM company
    ORDER BY company_id;
""")

rows = pg_cur.fetchall()
print("Total rows fetched:", len(rows))

companies = []
for row in rows:
    company_id, cik, ticker, company_name = row
    companies.append({
        "company_id": company_id,
        "cik": cik,
        "ticker": ticker,
        "company_name": company_name
    })

for c in companies[:5]:
    print(c)

In [ ]:
client = MongoClient("mongodb://localhost:27017")
db = client["sec_project"]
sec_filings = db["sec_filings"]

print("Connected to MongoDB: DB='sec_project', collection='sec_filings'")

In [ ]:
db.drop_collection("sec_filings")
sec_filings = db["sec_filings"]

sec_filings.create_index([("cik", 1), ("accession_number", 1)], unique=True)
sec_filings.create_index([("company_id", 1), ("filing_type", 1), ("filing_date", -1)])
sec_filings.create_index([("raw_text", "text")])

print("✅ Recreated collection 'sec_filings'")
print("Collection reset. Current count:", sec_filings.count_documents({}))

In [ ]:
SEC_API_HEADERS = {
    "User-Agent": "Yirui Chen yc4704@columbia.edu",
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov"
}

SEC_ARCHIVES_HEADERS = {
    "User-Agent": "Yirui Chen yc4704@columbia.edu",
    "Accept-Encoding": "gzip, deflate"
}

FIXED_START = date(2015, 11, 28)

def fetch_company_submissions(cik_10):
    url = f"https://data.sec.gov/submissions/CIK{cik_10}.json"
    resp = requests.get(url, headers=SEC_API_HEADERS)
    if resp.status_code != 200:
        print(f"❌ Failed submissions for {cik_10}, status={resp.status_code}")
        return None
    return resp.json()

def extract_filings_meta(sub_json, cik_10):
    if not sub_json:
        return []

    recent = sub_json["filings"]["recent"]
    forms = recent.get("form", [])
    dates = recent.get("filingDate", [])
    accs  = recent.get("accessionNumber", [])
    docs  = recent.get("primaryDocument", [])

    filings = []
    for form, d, acc, doc in zip(forms, dates, accs, docs):
        if form not in ("10-K", "10-Q","8-K"):
            continue
        try:
            y, m, da = map(int, d.split("-"))
            f_date = date(y, m, da)
        except:
            continue
        if f_date < FIXED_START:
            continue

        filings.append({
            "cik": cik_10,
            "filing_type": form,
            "filing_date": f_date,
            "accession_number": acc,
            "primary_doc": doc
        })

    return filings

In [ ]:
def build_filing_paths(cik_10, accession_number, primary_doc):
    cik_no_zero = str(int(cik_10))
    acc_nodash = accession_number.replace("-", "")

    base_dir = f"https://www.sec.gov/Archives/edgar/data/{cik_no_zero}/{acc_nodash}/"
    index_url = base_dir + f"{accession_number}-index.htm"
    main_doc_url = base_dir + primary_doc  # e.g. bkti20250930_10q.htm

    return base_dir, index_url, main_doc_url

def download_html(url):
    resp = requests.get(url, headers=SEC_ARCHIVES_HEADERS)
    if resp.status_code != 200:
        print(f"  ❌ Failed: {url} status={resp.status_code}")
        return None
    return resp.text

def html_to_text(html):
    soup = BeautifulSoup(html, "html.parser")
    for s in soup(["script", "style"]):
        s.decompose()
    lines = [line.strip() for line in soup.get_text("\n").splitlines()]
    return "\n".join(l for l in lines if l)


In [ ]:
def process_company_filings(company):
    company_id = company["company_id"]
    cik_10 = company["cik"].zfill(10)
    ticker = company["ticker"]
    name = company["company_name"]

    print(f"\n=== CIK={cik_10} ticker={ticker} ===")

    sub_json = fetch_company_submissions(cik_10)
    if sub_json is None:
        print("  ⚠️ No submissions JSON")
        return

    filings = extract_filings_meta(sub_json, cik_10)
    print(f"  Found {len(filings)} filings.")

    inserted = 0

    for meta in filings:
        base_dir, index_url, main_doc_url = build_filing_paths(
            cik_10,
            meta["accession_number"],
            meta["primary_doc"]
        )

        print(f"  -> {meta['filing_type']} {meta['filing_date']}")

        # Try main doc first
        html = download_html(main_doc_url)
        used_index = False

        if html is None:
            print("     ⚠️ main doc failed; trying index")
            html = download_html(index_url)
            if html is None:
                print("     ⛔ Skip filing")
                continue
            used_index = True

        text = html_to_text(html)

        doc = {
            "company_id": company_id,
            "cik": cik_10,
            "ticker": ticker,
            "company_name": name,

            "filing_type": meta["filing_type"],
            "filing_date": meta["filing_date"].isoformat(),
            "accession_number": meta["accession_number"],
            "primary_doc": meta["primary_doc"],

            "source_main": main_doc_url,
            "source_index": index_url,
            "used_index_page": used_index,

            "raw_text": text
        }

        try:
            sec_filings.insert_one(doc)
            inserted += 1
            print("     ✔ inserted")
        except Exception as e:
            print("     ⚠️ skipped:", e)

    print(f"  ✅ Done. Inserted {inserted}.")


In [ ]:
print("Total companies:", len(companies))

for idx, company in enumerate(companies, start=1):
    print(f"\n=== [{idx}/{len(companies)}] {company['ticker']} ===")
    process_company_filings(company)
    time.sleep(0.5)  

print("\n🎉 All companies processed!")

In [ ]:
print("db name:", db.name)
print("collection full name:", sec_filings.full_name)
print("docs:", sec_filings.count_documents({}))

In [ ]:
print("Total filings stored:", sec_filings.count_documents({}))

sample = sec_filings.find_one(sort=[("_id", -1)])
if sample:
    print("\nSample doc:", sample["ticker"], sample["filing_type"],
          sample["filing_date"], "| used_index =", sample["used_index_page"])
    print("\n--- Raw text preview ---\n")
    print(sample["raw_text"][:1500])

# Create filling_evet table

In [ ]:
import psycopg
import pandas as pd
import numpy as np
from pymongo import MongoClient
from datetime import datetime, timedelta
from flask import Flask, request, render_template_string

In [ ]:
import os
pg_conn = psycopg.connect(
    host="localhost",
    port="5432",
    dbname="final",
    user="postgres",
    password=os.environ["POSTGRES_PASSWORD"]
)
pg_cur = pg_conn.cursor()
print("Connected to PostgreSQL 'final'")

mongo_client = MongoClient("mongodb://localhost:27017")
mongo_db = mongo_client["sec_project"]
sec_filings = mongo_db["sec_filings"]
print("Connected to MongoDB 'sec_project.sec_filings'")

In [ ]:
pg_cur.execute("""
    DROP TABLE IF EXISTS filing_event;

    CREATE TABLE filing_event (
        event_id         SERIAL PRIMARY KEY,
        company_id       INTEGER NOT NULL REFERENCES company(company_id),
        event_date       DATE NOT NULL,
        event_type       VARCHAR(32) NOT NULL,   -- FILING_10K / FILING_10Q / FILING_8K / FILING_OTHER
        filing_type      VARCHAR(16),
        filing_accession VARCHAR(64),
        description      TEXT
    );

    CREATE INDEX idx_filing_event_company_date
        ON filing_event(company_id, event_date);
""")
pg_conn.commit()
print("Table 'filing_event' created.")

In [ ]:
df_companies = pd.read_sql(
    "SELECT company_id, ticker, company_name FROM company;",
    pg_conn
)
print("Total companies:", len(df_companies))

ticker_to_id = {
    row["ticker"].upper(): int(row["company_id"])
    for _, row in df_companies.iterrows()
}

In [ ]:
def map_event_type(form_type: str) -> str:
   
    if not form_type:
        return "FILING_OTHER"
    ft = form_type.upper()
    if "10-K" in ft:
        return "FILING_10K"
    if "10-Q" in ft:
        return "FILING_10Q"
    if "8-K" in ft:
        return "FILING_8K"
    return "FILING_OTHER"

In [ ]:
pg_cur.execute("TRUNCATE TABLE filing_event;")
pg_conn.commit()

inserted = 0
skipped_unknown_ticker = 0

cursor = sec_filings.find({}, {
    "_id": 0,
    "ticker": 1,
    "filing_date": 1,
    "filing_type": 1,
    "accession_number": 1
})

for doc in cursor:
    tk = (doc.get("ticker") or "").upper().strip()
    if not tk:
        continue

    company_id = ticker_to_id.get(tk)
    if company_id is None:
        skipped_unknown_ticker += 1
        continue

    filing_date_raw = doc.get("filing_date")
    if not filing_date_raw:
        continue

    if isinstance(filing_date_raw, str):
        try:
            filing_date = datetime.fromisoformat(filing_date_raw).date()
        except ValueError:
            try:
                filing_date = datetime.strptime(filing_date_raw, "%Y-%m-%d").date()
            except Exception:
                continue
    else:
        filing_date = filing_date_raw.date()

    filing_type = doc.get("filing_type") or ""
    event_type = map_event_type(filing_type)
    accession = doc.get("accession_number") or ""

    pg_cur.execute("""
        INSERT INTO filing_event
            (company_id, event_date, event_type, filing_type, filing_accession, description)
        VALUES
            (%s, %s, %s, %s, %s, %s);
    """, (
        company_id,
        filing_date,
        event_type,
        filing_type,
        accession,
        None
    ))
    inserted += 1

pg_conn.commit()
print("Inserted", inserted, "rows into filing_event.")
print("Skipped because ticker not in company table:", skipped_unknown_ticker)

Calculate the values of a certain company within ±window days around a certain event date
- Daily stock yield return
- Benchmark return of the technology sector
- abnormal_return = return - benchmark
return DataFrame: trade_date, day_offset, return, benchmark_return, abnormal_return

In [ ]:
from datetime import timedelta
import pandas as pd

def compute_event_window_returns(company_id, event_date, window=3):

    start_date = event_date - timedelta(days=window + 1)
    end_date   = event_date + timedelta(days=window)

    cur = pg_conn.cursor()

    cur.execute("""
        SELECT trade_date, close
        FROM daily_price
        WHERE company_id = %s
          AND trade_date BETWEEN %s AND %s
        ORDER BY trade_date;
    """, (company_id, start_date, end_date))
    rows = cur.fetchall()
    if len(rows) < 2:
        return None

    df = pd.DataFrame(rows, columns=["trade_date", "close"])
    df["trade_date"] = pd.to_datetime(df["trade_date"])
    df["close"] = df["close"].astype(float)
    df["return"] = df["close"].pct_change()

    cur.execute("""
        SELECT trade_date,
               AVG(ret)::float8 AS benchmark_return
        FROM (
            SELECT
                company_id,
                trade_date,
                close,
                close / NULLIF(
                    LAG(close) OVER (
                        PARTITION BY company_id
                        ORDER BY trade_date
                    ), 0
                ) - 1 AS ret
            FROM daily_price
            WHERE trade_date BETWEEN %s AND %s
        ) t
        WHERE ret IS NOT NULL
        GROUP BY trade_date
        ORDER BY trade_date;
    """, (start_date, end_date))
    bm_rows = cur.fetchall()

    df_bm = pd.DataFrame(bm_rows, columns=["trade_date", "benchmark_return"])
    if not df_bm.empty:
        df_bm["trade_date"] = pd.to_datetime(df_bm["trade_date"])
        df_bm["benchmark_return"] = df_bm["benchmark_return"].astype(float)

    df_merged = df.merge(df_bm, on="trade_date", how="left")
    df_merged["return"] = df_merged["return"].astype(float)
    df_merged["benchmark_return"] = df_merged["benchmark_return"].astype(float)
    df_merged["abnormal_return"] = df_merged["return"] - df_merged["benchmark_return"]

    event_ts = pd.to_datetime(event_date)
    df_merged["day_offset"] = (df_merged["trade_date"] - event_ts).dt.days

    df_win = df_merged[
        (df_merged["day_offset"] >= -window) &
        (df_merged["day_offset"] <= window)
    ].copy()

    return df_win

Take the event from filing_event + company according to the conditions and calculate it
- AR(0)
-CAR (sum of abnormal_return within the window)
- Volatility (standard deviation of abnormal_return within the window)
Add keyword: Search in MongoDB's raw_text and filter for accession
Return list[dict]

In [ ]:
def get_event_summaries(ticker=None, date=None, event_type=None, keyword=None, window=3, limit=20):

    valid_accessions = None
    if keyword:
        print(f"Searching MongoDB for keyword: '{keyword}' ...")
        cursor = sec_filings.find(
            {"raw_text": {"$regex": keyword, "$options": "i"}},
            {"accession_number": 1}
        )
        valid_accessions = [doc["accession_number"] for doc in cursor if doc.get("accession_number")]
        print(f"   Found {len(valid_accessions)} matching filings in MongoDB.")
        
        if not valid_accessions:
            return []

    sql = """
        SELECT fe.event_id,
               fe.event_date,
               fe.event_type,
               fe.filing_type,
               fe.filing_accession,
               fe.company_id,
               c.ticker,
               c.company_name
        FROM filing_event fe
        JOIN company c ON fe.company_id = c.company_id
    """
    conds = []
    params = []

    if ticker:
        conds.append("c.ticker = %s")
        params.append(ticker.upper())

    if date:
        conds.append("fe.event_date = %s")
        params.append(date)

    if event_type and event_type != "ALL":
        conds.append("fe.event_type = %s")
        params.append(event_type)

    if valid_accessions is not None:
        conds.append("fe.filing_accession = ANY(%s)")
        params.append(valid_accessions)

    if conds:
        sql += " WHERE " + " AND ".join(conds)

    sql += " ORDER BY fe.event_date DESC LIMIT %s;"
    params.append(limit)

    pg_cur.execute(sql, params)
    rows = pg_cur.fetchall()

    results = []
    for (event_id, event_date, ev_type, filing_type, accession,
         company_id, tk, company_name) in rows:

        df_win = compute_event_window_returns(company_id, event_date, window)
        if df_win is None or df_win.empty:
            ar0 = None
            car = None
            vol = None
        else:
            row0 = df_win[df_win["day_offset"] == 0]
            if not row0.empty and not pd.isna(row0["abnormal_return"].iloc[0]):
                ar0 = float(row0["abnormal_return"].iloc[0])
            else:
                ar0 = None

            car = float(df_win["abnormal_return"].sum()) if "abnormal_return" in df_win else None

            vol = float(df_win["abnormal_return"].std(ddof=1)) if "abnormal_return" in df_win else None

        sec_url = ""
        if accession:
            doc = sec_filings.find_one(
                {"accession_number": accession},
                {"_id": 0, "source_main": 1}
            )
            if doc and doc.get("source_main"):
                sec_url = doc["source_main"]

        if not sec_url:
            doc = sec_filings.find_one(
                {"ticker": tk.upper(), "filing_date": event_date.isoformat()},
                {"_id": 0, "source_main": 1}
            )
            if doc and doc.get("source_main"):
                sec_url = doc["source_main"]

        results.append({
            "event_id": event_id,
            "ticker": tk,
            "company_name": company_name,
            "event_date": event_date.isoformat(),
            "event_type": ev_type,
            "filing_type": filing_type,
            "ar0": ar0,
            "car": car,
            "vol": vol,
            "sec_url": sec_url,
        })

    return results

In [ ]:
PAGE_TEMPLATE = """
<!doctype html>
<html>
  <head>
    <meta charset="utf-8" />
    <title>Event Study Dashboard</title>
    <style>
      body { font-family: Arial, sans-serif; background-color: #fafafa; }
      h1 { text-align: center; margin-top: 20px; }

      .panel {
        border-radius: 8px;
        padding: 16px 24px;
        margin: 16px auto;
        width: 900px;
        background-color: #fffbe6; /* light yellow */
        box-shadow: 0 0 4px rgba(0,0,0,0.1);
      }
      .panel-title {
        text-align: center;
        font-weight: bold;
        margin-bottom: 10px;
      }
      .input-row {
        display: flex;
        gap: 20px;
        justify-content: center;
        margin-top: 8px;
        flex-wrap: wrap;
      }
      .input-box {
        background-color: #f5f0ff;
        padding: 8px 12px;
        border-radius: 6px;
        border: 1px solid #d0c4ff;
      }
      label {
        margin-right: 4px;
        font-size: 14px;
      }
      input[type="text"], input[type="number"], input[type="date"], select {
        border: none;
        border-bottom: 1px solid #888;
        background-color: transparent;
        outline: none;
        padding: 2px 4px;
      }
      .buttons {
        display: flex;
        justify-content: center;
        gap: 40px;
        margin-top: 10px;
      }
      .btn-main {
        padding: 8px 24px;
        border-radius: 6px;
        border: none;
        font-size: 16px;
        cursor: pointer;
      }
      .btn-exec {
        background-color: #28a745;
        color: #fff;
      }
      .btn-reset {
        background-color: #f0f0f0;
        color: #333;
      }

      table {
        border-collapse: collapse;
        width: 100%;
        margin-top: 10px;
      }
      th, td {
        border: 1px solid #aaa;
        padding: 4px 6px;
        font-size: 13px;
        text-align: center;
      }
      th {
        background-color: #f0f0f0;
      }
      .sec-link a {
        color: #3366cc;
        text-decoration: none;
      }
      .sec-link a:hover {
        text-decoration: underline;
      }
      .no-result {
        font-size: 13px;
        color: #666;
        text-align: center;
        margin-top: 10px;
      }
    </style>
  </head>
  <body>
    <h1>Event Study Dashboard</h1>

    <!-- INPUT PANEL -->
    <form method="get" action="/">
      <div class="panel">
        <div class="panel-title">Input Panel</div>
        <div class="input-row">
          <div class="input-box">
            <label>Ticker:</label>
            <input type="text" name="ticker" value="{{ ticker }}" placeholder="e.g. AAPL">
          </div>
          <div class="input-box">
            <label>Date:</label>
            <!-- HTML5 date input with calendar -->
            <input type="date" name="date" value="{{ date }}">
          </div>
          <div class="input-box">
            <label>Event Type:</label>
            <select name="event_type">
              <option value="ALL" {% if event_type == "ALL" %}selected{% endif %}>All Types</option>
              <option value="FILING_10K" {% if event_type == "FILING_10K" %}selected{% endif %}>10-K</option>
              <option value="FILING_10Q" {% if event_type == "FILING_10Q" %}selected{% endif %}>10-Q</option>
              <option value="FILING_8K" {% if event_type == "FILING_8K" %}selected{% endif %}>8-K</option>
              <option value="FILING_OTHER" {% if event_type == "FILING_OTHER" %}selected{% endif %}>Other</option>
            </select>
          </div>
          <div class="input-box">
            <label>Window (days):</label>
            <input type="number" name="window" min="1" max="30" value="{{ window }}">
          </div>
        </div>
        
        <!-- New Row for Keyword Search -->
        <div class="input-row">
            <div class="input-box" style="width: 60%;">
                <label>Keyword (MongoDB Search):</label>
                <input type="text" name="keyword" value="{{ keyword }}" placeholder="Search in filing text..." style="width: 80%;">
            </div>
        </div>
      </div>

      <!-- ACTION PANEL -->
      <div class="panel">
        <div class="panel-title">Actions</div>
        <div class="buttons">
          <button class="btn-main btn-exec" type="submit">Run Study</button>
          <button class="btn-main btn-reset" type="button" onclick="window.location='/'">Reset</button>
        </div>
      </div>
    </form>

    <!-- RESULT PANEL -->
    <div class="panel">
      <div class="panel-title">Results</div>
      {% if events and events|length > 0 %}
        <table>
          <tr>
            <th>Ticker</th>
            <th>Date</th>
            <th>Type</th>
            <th>AR(0)</th>
            <th>CAR</th>
            <th>Volatility</th>
            <th>Action</th>
          </tr>
          {% for e in events %}
          <tr>
            <td>{{ e.ticker }}</td>
            <td>{{ e.event_date }}</td>
            <td>{{ e.filing_type or e.event_type }}</td>
            <td>
              {% if e.ar0 is not none %}
                {{ "%.2f"|format(e.ar0 * 100) }}%
              {% else %}
                -
              {% endif %}
            </td>
            <td>
              {% if e.car is not none %}
                {{ "%.2f"|format(e.car * 100) }}%
              {% else %}
                -
              {% endif %}
            </td>
            <td>
              {% if e.vol is not none %}
                {{ "%.2f"|format(e.vol * 100) }}%
              {% else %}
                -
              {% endif %}
            </td>
            <td class="sec-link">
              {% if e.sec_url %}
                <a href="{{ e.sec_url }}" target="_blank">View SEC Filing</a>
              {% else %}
                No link
              {% endif %}
            </td>
          </tr>
          {% endfor %}
        </table>
      {% else %}
        <div class="no-result">
          No events found for the current filters.
          {% if keyword %}
            <br>(Keyword search '{{ keyword }}' returned no matches in MongoDB)
          {% endif %}
        </div>
      {% endif %}
    </div>
  </body>
</html>
"""

In [ ]:
app = Flask(__name__)

In [ ]:
@app.route("/", methods=["GET"])
def index():
    ticker = (request.args.get("ticker") or "").strip().upper()
    date_str = (request.args.get("date") or "").strip()
    event_type = request.args.get("event_type", "ALL")
    window_str = request.args.get("window", "3")
    keyword = (request.args.get("keyword") or "").strip()

    try:
        window = int(window_str)
        if window < 1:
            window = 1
        if window > 30:
            window = 30
    except ValueError:
        window = 3

    date_val = None
    if date_str:
        try:
            date_val = datetime.strptime(date_str, "%Y-%m-%d").date()
        except ValueError:
            date_val = None

    events = get_event_summaries(
        ticker=ticker if ticker else None,
        date=date_val,
        event_type=event_type,
        keyword=keyword if keyword else None,
        window=window,
        limit=20
    )

    return render_template_string(
        PAGE_TEMPLATE,
        ticker=ticker,
        date=date_str,
        event_type=event_type,
        window=window,
        keyword=keyword,
        events=events
    )

In [ ]:
app.run(debug=True, use_reloader=False)

# Data Size Check

In [ ]:
stats = db.command("collstats", "sec_filings")

print("=== MongoDB Collection Size ===")
print(f"Documents count         : {stats['count']}")
print(f"Raw data size           : {stats['size'] / 1024**3:.3f} GB")
print(f"Actual storage size     : {stats['storageSize'] / 1024**3:.3f} GB")
print(f"Total index size        : {stats['totalIndexSize'] / 1024**3:.3f} GB")


In [ ]:
import os
import psycopg

conn = psycopg.connect(
    host="localhost",
    port="5432",
    dbname="final",
    user="postgres",
    password=os.environ["POSTGRES_PASSWORD"]
)
cur = conn.cursor()

tables_to_show = ["daily_price", "filing_event", "company"]

cur.execute("""
    SELECT 
        relname AS table_name,
        pg_size_pretty(pg_total_relation_size(relid)) AS total_size
    FROM pg_catalog.pg_statio_user_tables
    ORDER BY pg_total_relation_size(relid) DESC;
""")

print("\nSize of selected tables:")
for table_name, size in cur.fetchall():
    if table_name in tables_to_show:
        print(f"{table_name:20s} {size}")